In [4]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str):
    print(title)
    # 打印列名
    header = " " * 10 + "".join(c.rjust(10) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(10)
        for val in row:
            row_str += colorize(val, width=10)
        print(row_str)
    print()


# ================== 主逻辑 ==================
# df = pd.read_csv("./history_results/results8/result_summary.csv")
df = pd.read_csv("./results/result_summary.csv")

loss_list = [f"loss{i}" for i in range(1, 6)]
# loss_list = [f"loss{i}" for i in [3,5]]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 10"]

# ========== 生成表 (平均提升百分比) ==========
def make_tables(df, label):
    for topk in topk_list:
        df_topk = df[df["TopK"] == topk]
        pivot = (
            df_topk[df_topk["Loss"].isin(loss_list)]
            .groupby(["Model", "Loss"])["RelDiff(%)"]
            .mean()
            .reset_index()
        )
        table = pivot.pivot(index="Loss", columns="Model", values="RelDiff(%)").round(3)
        print_colored_table(table, f"\n=== {label} | {topk} ===")


print("========== 汇总表 ==========")

# 遍历 campus
for campus, df_c in df.groupby("Campus"):
    print(f"\n########## Campus: {campus} ##########")
    make_tables(df_c, f"Results | Campus={campus}")


========== 汇总表 ==========

########## Campus: campus_10 ##########

=== Results | Campus=campus_10 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.386    -1.922    -0.301    -0.262
loss2         -0.584    -0.982     0.746     0.099
loss3          0.589     0.346     0.851     0.441
loss4         -3.420    -4.342    -1.573    -2.135
loss5          0.111     0.073     1.054     0.193


########## Campus: campus_102 ##########

=== Results | Campus=campus_102 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -3.660    -2.729    -1.969    -2.628
loss2         -0.073    -0.912     0.091    -0.158
loss3          0.073     0.418     0.953    -0.267
loss4         -3.293    -2.964    -1.074    -2.529
loss5          0.031     0.131     0.386    -0.240


########## Campus: campus_143 ##########

=== Results | Campus=campus_143 | Top 10 ===
                 NCL       SGL    SimGCL   XSimGCL
loss1         -1.079    -0.366     1.093     1.

In [5]:
import pandas as pd

# ================== 工具函数 ==================
# 颜色渲染：正数红色，其余默认
def colorize(val: float, width=10):
    if pd.isna(val):
        return " " * width
    s = f"{val:.3f}".rjust(width)  # 固定宽度，保证对齐
    if val > 0:
        return f"\033[91m{s}\033[0m"  # 红色
    return s

# 表格打印（按列宽对齐 + 颜色）
def print_colored_table(df: pd.DataFrame, title: str, value_width: int = 10):
    print(title)
    # 动态确定行名列宽
    row_label_width = max(12, max((len(str(i)) for i in df.index), default=0) + 2)
    # 打印列名
    header = " " * row_label_width + "".join(str(c).rjust(value_width) for c in df.columns)
    print(header)
    # 打印每行
    for idx, row in df.iterrows():
        row_str = str(idx).ljust(row_label_width)
        for val in row:
            row_str += colorize(val, width=value_width)
        print(row_str)
    print()


# ================== 主逻辑 ==================
# df = pd.read_csv("./history_results/results8/result_summary.csv")
df = pd.read_csv("./results/result_summary.csv")

# 确保数值列为数值类型
df["RelDiff(%)"] = pd.to_numeric(df["RelDiff(%)"], errors="coerce")

# 只考虑 loss3 和 loss5
loss_keep = ["loss3", "loss5"]
# topk_list = ["Top 3", "Top 5", "Top 10", "Top 20"]
topk_list = ["Top 3", "Top 5", "Top 10"]

print("========== 汇总表 ==========")

# 过滤数据
subset = df[df["Loss"].isin(loss_keep)]

# 遍历 (Campus, Loss, TopK)，分别打印表格
for campus, df_c in subset.groupby("Campus", sort=True):
    print(f"\n########## Campus: {campus} ##########")
    for loss in loss_keep:
        print(f"###### Loss: {loss} ######")
        df_l = df_c[df_c["Loss"] == loss]
        for topk in topk_list:
            g = df_l[df_l["TopK"] == topk]
            if g.empty:
                continue
            pivot = (
                g.groupby(["Metric", "Model"])["RelDiff(%)"]
                 .mean()
                 .reset_index()
                 .pivot(index="Metric", columns="Model", values="RelDiff(%)")
                 .sort_index()
                 .round(3)
            )
            pivot = pivot.reindex(sorted(pivot.columns), axis=1)  # 模型列排序
            print_colored_table(
                pivot,
                f"=== Results | Campus={campus} | Loss={loss} | TopK={topk} ==="
            )


========== 汇总表 ==========

########## Campus: campus_10 ##########
###### Loss: loss3 ######
=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 3 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.643     2.780     0.166     1.479
NDCG             1.149     1.634     0.681     1.608
Precision        0.645     2.781     0.162     1.477
Recall           1.230     1.239     0.262     1.231

=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 5 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.023     1.125     0.287     0.410
NDCG             0.700     0.761     0.876     0.936
Precision        0.027     1.124     0.288     0.413
Recall           0.357     0.844     0.740     0.251

=== Results | Campus=campus_10 | Loss=loss3 | TopK=Top 10 ===
                   NCL       SGL    SimGCL   XSimGCL
Hit Ratio        0.528     0.339     0.975     0.234
NDCG             0.812     0.352     0.961     0.865
Precision        0.530     0.335

In [6]:
import pandas as pd
from colorama import Fore, Style

for model, subdf in df.groupby("Model"):
    print(f"\n===== Model: {model} =====")

    # 先转成字符串表格
    table_str = subdf.to_string(index=False)

    # 按行拆分
    lines = table_str.split("\n")

    # 第一行是表头，原样打印
    print(lines[0])

    # 从第二行开始逐行处理
    for i, (_, row) in enumerate(subdf.iterrows(), start=1):
        line = lines[i]
        if row["RelDiff(%)"] > 0:
            print(Fore.RED + line + Style.RESET_ALL)
        else:
            print(line)



===== Model: NCL =====
Model     Campus  Loss   TopK    Metric  Baseline(loss0)   Value  AbsDiff  RelDiff(%)
  NCL  campus_10 loss1  Top 3 Hit Ratio          0.21929 0.21182 -0.00747   -3.406448
  NCL  campus_10 loss1  Top 3 Precision          0.27121 0.26198 -0.00923   -3.403267
  NCL  campus_10 loss1  Top 3    Recall          0.29504 0.28427 -0.01077   -3.650352
  NCL  campus_10 loss1  Top 3      NDCG          0.35425 0.34102 -0.01323   -3.734651
  NCL  campus_10 loss1  Top 5 Hit Ratio          0.30348 0.29378 -0.00970   -3.196257
  NCL  campus_10 loss1  Top 5 Precision          0.22520 0.21800 -0.00720   -3.197158
  NCL  campus_10 loss1  Top 5    Recall          0.38087 0.37067 -0.01020   -2.678079
  NCL  campus_10 loss1  Top 5      NDCG          0.37436 0.36195 -0.01241   -3.314991
  NCL  campus_10 loss1 Top 10 Hit Ratio          0.43723 0.43163 -0.00560   -1.280790
  NCL  campus_10 loss1 Top 10 Precision          0.16223 0.16015 -0.00208   -1.282130
  NCL  campus_10 loss1 Top 10 